# GeoCLEWs

**Revised code:** [Yalda Saedi](https://github.com/Ysaedi) <br />
**Supervision:** [Taco Niet](https://github.com/tniet) <br />
**Funding** [Mitacs](https://www.mitacs.ca/en) and [Catalyste+](https://www.catalysteplus.org/)

---------------------------

## Summary

GeoCLEWs is a versatile open-source script that offers a wide range of useful features for both developers and users. The script streamlines the detailed land and water processing steps required for Climate Land Energy Water systems (CLEWs) modelling. It is designed to efficiently collect, modify, and process  high-resolution land and water data from Global Agro-ecological Zones (GAEZ v5) database in an automated manner. GeoCLEWs processes agro-climatic potential yield, crop water deficit, crop evapotranspiration, precipitation, and land cover datasets.


This notebook builds upon the [CLEWs GIS Processing notbook](https://github.com/akorkovelos/un-clews-gis-work/blob/main/CLEWs%20GIS%20Processing.ipynb), developed by [Alexandros Korkovelos](https://github.com/akorkovelos), under the supervision of [Abhishek Shivakumar](https://github.com/abhishek0208) & Thomas Alfstad. Please note that the original notebook is currently non-functional due to its incompatibility with the latest version of the GAEZ database. Although the original code served as a valuable starting point, it has undergone significant revisions and improvements:

- Utilizing high-resolution GAEZ v5 datasets to detail land and water systems.
- Implementing customized geographical aggregation and Agglomerative Hierarchical clustering to capture cross-regional interdependencies and streamline computational complexity.
- Automating data collection, preparation, processing, and result generation for diverse geographical regions, reducing manual effort and minimizing human errors in WEF assessment tasks.
- Generating clewsy-compatible outputs.

At the beginning of the script, users can input and tailor the configuration to align with their project's unique requirements. Once the necessary inputs are provided, GeoCLEWs will automatically execute the subsequent steps, including the data collection from FAOSTAT (Food and Agriculture Organization of the United Nation) and retrieving high-resolution raster files from GAEZ v5. Additionally, it implements geographical and spatial clustering to detect cross-regional agro-ecological similarities and generate detailed land and water statistics.

This notebook performs six main analytical processes:

- **Part 1**: Initialization and configuration.
- **Part 2**: FAOSTAT and GAEZ data collection and preparation.
- **Part 3**: Generating land cells.
- **Part 4**: Geospatial attributes extraction to land cells.
- **Part 5**: Spatial clustering using agglomerative hierarchical clustering
- **Part 6**: Calculating key summary statistics generate outputs for further use in CLEWs modelling.
- **Part 7**: Generating clusters' boundaries shapefiles (if needed for further analysis).

Each part below is accompanied by a more detailed explanation of the involved processing steps.

# Part 1 : Initialization and Configuration

# 1.1. Importing Essential Modules 

To begin, it is necessary to import the required modules/libraries. For more information on the environment setup, please consult the README file. 

In [ ]:
# Importing necessary Python modules or libraries

# Numerical
import numpy as np
import pandas as pd
import requests

# Spatial
import geopandas as gpd
import rasterio
from rasterstats import zonal_stats
from geojson import Feature, Point, FeatureCollection
import json
from shapely.geometry import Polygon, Point
#import gdal
from pyproj import CRS
from rasterio.mask import mask
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from sklearn.cluster import AgglomerativeClustering


#Plotting
import plotly.graph_objects as go
import plotly.express as px
from IPython.display import display, Markdown, HTML
import matplotlib.pyplot as plt
%matplotlib inline
import warnings
from sklearn.cluster import KMeans

# System & Other
import time
import os
import datetime
import errno
import tempfile
import re
from pathlib import Path


start_time = time.time()


# 1.2. User Configuration  
This is the only part where the user needs to input values. The rest of the process will be automatically run based on the provided inputs. The code is designed with flexibility, allowing users to make changes and take advantage of customized settings at any time during the execution.


In [ ]:
# Provide specifications for the project 

Full_name = "Zambia"                   # Specify the full name of the country you wish to analyze (e.g. Zambia).
FAOSTAT_statistics = 2023              #Specify the year of the FAOSTAT statistics. Statistics are updated annually: https://www.fao.org/faostat/en/#data/QCL
number_of_main_crops = 4               #specify the number of main crops
total_crops = 6                        #Total number of crops to be considered (e.g if the total is 10 and 5 are main, the other 5 will fall under the cathegory "others")

SSP = "historical"                        # Choose your preferred SSP (Shared Socioeconomic Pathway) from the available options: historical, ssp126, ssp370, ssp585.
Climate_model = "AGERA5"                  #If ssp historical the only climate model is AGERA5. For all the other SSPs, the available climate models are: GFDL-ESM4, IPSL-CM6A-LR, MPI-ESM1-2-HR, MRI-ESM2-0, UKESM1-0-LL.


admin_level = 0                       # 0 for national processing, admin level 0.
                                      # 1 for processing at admin level 1.


#Aggregating regions based on administrative boundaries    
aggregate = True                      # If the admin_level is 1 or higher, you can select specific regions, group them together, and generate Agro-Ecological statistics for these regions as a single aggregated area.                            
                                      # It is crucial that the administrative boundary of the selected regions matches the specified admin_level and input shapefile. 
                                      # Set "False" to disable aggregation and "True" to enable aggregation.
aggregate_region = ["ZMB"]      # To select regions for aggregation, input the first 3 letters of each region's name, e.g., "TAI" for Taita Taveta county and "TUR" for Turkana county in Kenya, which can be grouped if admin_level=1.
                      
                                      #Aggregated regions are represented by the Grouped Region Cluster (GRC).

                         


## Coordinate and projection systems
crs_WGS84 = CRS("EPSG:4326")          # For the analysis, the original WGS84 coordinate system is used. 
crs_proj = CRS("EPSG:32735")          # Please provide proper projection system based on the geographical location of the selected country. 
                                      # In this example EPSG 32735 refers to a specific projection system for Zambia.
                                      # More info: http://epsg.io/ 

## 1.3. Directory Initialization and Structure

For easier replication of the code you may used the following directory structure:

* **~root/Data/input**    (a directory where your input data)
* **~root/global_raster_input**   (directory for GAEZ raster layers with global coverage.  Precipitation and land cover rasters have already been downloaded in the global_raster_input while agro-climatic potential yield, crop water deficit, and crop evapotranspiration with global coverage will be downloaded automatically in the following steps based on user input)
* **~root/cropped_raster_input** (a directory for cropped GAEZ global raster data based on administraty boundry of selected country to reduce computational complexity)

Results will be store in two automatically generated directories:
* **~root/Data/output**   (directory for general output)
* **~root/Data/output/summary_stats**   (a directory where the tabular outputs data and graphs are stored)

**Note!** In case you decide to use a different structure please revise the code below.

In [ ]:
# Directories
ROOT_DIR = os.path.abspath(os.curdir)
data_folder = "Data"

in_path = os.path.join(ROOT_DIR, data_folder + "\\"+ 'input')
in_path_raster = os.path.join(ROOT_DIR, 'global_raster_input')
out_path_raster = os.path.join(ROOT_DIR, 'cropped_raster_input')

if not os.path.exists(out_path_raster):
    try:
        os.makedirs(out_path_raster)
    except OSError as exc: 
        if exc.errno != errno.EEXIST:
            raise

out_path = os.path.join(ROOT_DIR, data_folder + "\\"+ 'output')
if not os.path.exists(out_path):
    try:
        os.makedirs(out_path)
    except OSError as exc: 
        if exc.errno != errno.EEXIST:
            raise
            
            
summary_stats_path = out_path + "\\" + "summary_stats"
if not os.path.exists(summary_stats_path):
    try:
        os.makedirs(summary_stats_path)
    except OSError as exc:                                        
        if exc.errno != errno.EEXIST:
            raise


out_path_dendrogram =out_path + "\\" + "dendrogram_graph"
if not os.path.exists(out_path_dendrogram):
    try:
        os.makedirs(out_path_dendrogram)
    except OSError as exc:                                         # Guard against race condition
        if exc.errno != errno.EEXIST:
            raise
            
out_path_elbow = out_path + "\\" + "elbow_graph"
if not os.path.exists(out_path_elbow):
    try:
        os.makedirs(out_path_elbow)
    except OSError as exc:                                         # Guard against race condition
        if exc.errno != errno.EEXIST:
            raise
            
out_path_clustering = out_path + "\\" + "spatial_clustering"
if not os.path.exists(out_path_clustering):
    try:
        os.makedirs(out_path_clustering)
    except OSError as exc:                                         # Guard against race condition
        if exc.errno != errno.EEXIST:
            raise
            

# 3 letter ISO code of the selected country
code = pd.read_csv('Country_code.csv')                            # More info: https://www.nationsonline.org/oneworld/country_code_list.htm                                        
code_name = code[code['Full_name']== Full_name]
country_name = code_name.iloc[0]['country_code']            
            
# supporting vector point name
shp_nm = "{}_data.shp".format(country_name)
    
#administrative boundary
admin0_nm = '{}_adm0.shp'.format(country_name)                     # administrative boundaries - national analysis

# Name of final result file
output_nm = "{}_vector_admin{}_land_cells".format(country_name, admin_level)
result_nm = "{}_vector_admin{}_land_cells_with_attributes".format(country_name, admin_level)

# Part 2 : FAOSTAT and GAEZ Data Collection and Preparation

## 2.1. FAOSTAT Collection and Preparation
Finding top crops in terms of harvested area from the latest statistical database provided by Food and Agriculture Organization of the United Nations (FAOSTAT).

### 2.1.1. Retrieve Top 10 Crops

In [ ]:
# Read the FAOSTAT file
data = pd.read_csv(f'FAOSTAT_{FAOSTAT_statistics}.csv')
filtered_data = data[data['Area'] == Full_name]                

# Sorting based on the harvested area in descending order and get top rows
# Retrieve data according to the user-defined country
top_10_values = filtered_data.nlargest(total_crops, 'Value')
all_crops = top_10_values['Item'].tolist() 

main_crops= all_crops[:number_of_main_crops]
other_crops = all_crops[number_of_main_crops:]

display(Markdown(f" **Top {number_of_main_crops} crops considering harvested area:** {main_crops}"))
display(Markdown(f" **Crops ranked in the top {total_crops} FAO dataset are:** {other_crops}"))

### 2.1.2. FAOSTAT Standardizing

In [ ]:
# FAO correction: 3 letter naming convention per crop considering CLEWs naming format

Crop_code = pd.read_csv('Crop_code.csv')
crop_name = []  

for item in main_crops:
    matching_rows = Crop_code[Crop_code['Name']==item]
    
    if not matching_rows.empty:
        crop_name.extend(matching_rows['Code'].tolist())  

other_crop_name = []  

for item in other_crops:
    matching_rows = Crop_code[Crop_code['Name']==item]
    
    if not matching_rows.empty:
        other_crop_name.extend(matching_rows['Code'].tolist())  


#Adding "prc" refering to annual precipitation
crop_name = crop_name + ["prc"]

display(Markdown(' **Based on 3-letter naming, the main crop list from the FAOSTAT is :** {}'.format(crop_name)))
display(Markdown(' **Based on 3-letter naming, additional crop list from the FAOSTAT is :** {}'.format(other_crop_name)))




## 2.2. GAEZ Data Collection and Preparation

GeoCLEWs collects TIFF data from the Global Agro-Ecological Zones data portal for the following variables: agro-climatic potential yield, crop water deficit, and crop evapotranspiration. Precipitation, and land cover have already downloaded in the directory. The GAEZ v5 rasters are saved in a new repository called GAEZ_v5_data.

### 2.2.1. GAEZ Data Acquisition

Each dataset of the GAEZ v5 has a link to a Google Cloud Storage. For more information follow https://github.com/un-fao/gaezv5/wiki/21.-GAEZ-v5-Platform-User-Guide 

In [ ]:
base_url = 'https://storage.googleapis.com/fao-gismgr-gaez-v5-data/DATA/GAEZ-V5/MAPSET/RES02-'

GAEZ_folder = os.path.join(ROOT_DIR, 'GAEZ_v5_data')
if not os.path.exists(GAEZ_folder):
    try:
        os.makedirs(GAEZ_folder)
    except OSError as exc:
        if exc.errno != errno.EEXIST:
            raise

#Data filtering based on User defined SSP and Climate model
if str(SSP).lower().startswith('hist') or str(SSP).lower() == 'historical':
    period_code = 'HP0120'
    ssp_code = 'HIST'
    climate_models = [Climate_model]
else:
    period_code = 'FP2140'
    ssp_map = {'ssp126': 'SSP126', 'ssp370': 'SSP370', 'ssp585': 'SSP585'}
    ssp_code = ssp_map.get(str(SSP).lower(), str(SSP).upper())
    climate_models = [Climate_model]


#Mechanization and farming types: HI = High irrigated, LI = Low irrigated, LR = Low Rain-fed, HR = High rainfed
water_codes = ['HILM', 'LILM', 'LRLM', 'HRLM']

def make_rows(var_tag):
    rows = []
    selected_crops = [c for c in crop_name if str(c).lower() != 'prc']
    for climate in climate_models:
        for crop_code in selected_crops:
            rc_rows = Crop_code[Crop_code['Code'] == crop_code]
            if not rc_rows.empty:
                crop_label = rc_rows.iloc[0]['Name']
            else:
                crop_label = crop_code

            for variable in ['YLD', 'WDE', 'ETA']:
                new_base = base_url + f'{variable}/'
                for input in water_codes:
                    input_level = 'High' if input in ('HILM', 'HRLM') else 'Low'
                    filename = f'GAEZ-V5.RES02-{variable}.{period_code}.{climate}.{ssp_code}.{crop_code}.{input}.tif'
                    url = new_base + filename
                    rows.append({
                        'Name': filename,
                        'Path to File': os.path.join('\\', 'MAPSET', filename),
                        'Variable Name': variable,
                        'SSP': SSP,
                        'Crop': crop_label,
                        'Input Level': input_level,
                        'Download URL': url,
                        'New Crop': crop_code
                    })
      
        return pd.DataFrame(rows) 

for var_tag in ['YLD', 'WDE', 'ETA']:
    if var_tag == 'YLD':
        yld_df = make_rows('YLD')
    if var_tag == 'WDE':
        cwd_df = make_rows('WDE')
    if var_tag == 'ETA':
        evt_df = make_rows('ETA')

yld_High_input = yld_df[yld_df['Input Level'] == 'High']
yld_Low_input = yld_df[yld_df['Input Level'] == 'Low']
cwd_High_input = cwd_df[cwd_df['Input Level'] == 'High']
cwd_Low_input = cwd_df[cwd_df['Input Level'] == 'Low']
evt_High_input = evt_df[evt_df['Input Level'] == 'High']
evt_Low_input = evt_df[evt_df['Input Level'] == 'Low']

In [ ]:
#Download main crops
def download_URL(dataframe, column, folder_name):
    os.makedirs(folder_name, exist_ok=True)
    for index, row in dataframe.iterrows():
        url = str(row.get(column, '')).strip()
        if not url or url.lower() in ('nan', 'none'):
            print(f"No URL for row {index}, skipping")
            continue
        try:
            head = requests.head(url, allow_redirects=True, timeout=10)
            if head.status_code != 200:
                print(f"URL not available (status {head.status_code}): {url}")
                continue
        except Exception as e:
            print(f"HEAD request failed for {url}: {e}")
            continue

        fname = os.path.basename(url.split('?')[0])
        if not fname.lower().endswith('.tif') and not fname.lower().endswith('.tiff'):
            fname = f"{row.get('New Crop','unk')}_{row.get('Variable Name','var')}_{row.get('Input Level','lvl')}.tif"

        file_path = os.path.join(folder_name, fname)
        if os.path.exists(file_path):
            print(f"Exists: {fname}")
            continue

        try:
            with requests.get(url, stream=True, timeout=60) as r:
                r.raise_for_status()
                with open(file_path, 'wb') as f:
                    for chunk in r.iter_content(chunk_size=8192):
                        if chunk:
                            f.write(chunk)
            print(f"Downloaded: {fname}")
        except Exception as e:
            print(f"Failed to download {url}: {e}")

print("Starting downloads...")
download_URL(yld_High_input, 'Download URL', GAEZ_folder )
download_URL(yld_Low_input, 'Download URL', GAEZ_folder)
download_URL(cwd_High_input, 'Download URL', GAEZ_folder)
download_URL(evt_High_input, 'Download URL', GAEZ_folder)
download_URL(cwd_Low_input, 'Download URL', GAEZ_folder)
download_URL(evt_Low_input, 'Download URL', GAEZ_folder)
print("Downloads complete.")

In [ ]:
#Create other crop rasters based on the the Other_crop list
def process_other_crops(folder_name):
    """
    Downloads individual 'other' crops to a temporary directory, 
    calculates their cell-by-cell average, saves the result as an 'OTHR' raster,
    and automatically cleans up the individual source files.
    """
    if not other_crop_name:
        print("No crops found in 'other_crop_name'. Skipping OTHR generation.")
        return

    print("\n--- Starting  ---")
    
    # Use a secure, self-cleaning temporary directory for individual downloads
    with tempfile.TemporaryDirectory() as temp_dir:
        
        for variable in ['YLD', 'WDE', 'ETA']:
            new_base = base_url + f'{variable}/'
            
            for climate in climate_models:
                for wc in water_codes:
                    # Construct the final target OTHR filename
                    othr_filename = f'GAEZ-V5.RES02-{variable}.{period_code}.{climate}.{ssp_code}.OTHR.{wc}.tif'
                    othr_file_path = os.path.join(folder_name, othr_filename)

                    # Skip if this OTHR file was already generated in a previous run
                    if os.path.exists(othr_file_path):
                        print(f"Exists: {othr_filename}")
                        continue
                    
                    valid_arrays = []
                    raster_profile = None
                    
                    # 1. Download each individual "other" crop for this exact scenario
                    for crop_code in other_crop_name:
                        filename = f'GAEZ-V5.RES02-{variable}.{period_code}.{climate}.{ssp_code}.crop_code.{wc}.tif'
                        # Fix string substitution for the URL
                        filename = f'GAEZ-V5.RES02-{variable}.{period_code}.{climate}.{ssp_code}.{crop_code}.{wc}.tif'
                        url = new_base + filename
                        temp_file_path = os.path.join(temp_dir, filename)
                        
                        try:
                            # Verify URL existence
                            head = requests.head(url, allow_redirects=True, timeout=10)
                            if head.status_code != 200:
                                continue
                            
                            # Stream download to temporary folder
                            with requests.get(url, stream=True, timeout=60) as r:
                                r.raise_for_status()
                                with open(temp_file_path, 'wb') as f:
                                    for chunk in r.iter_content(chunk_size=8192):
                                        if chunk:
                                            f.write(chunk)
                            
                            # 2. Read the raster data into memory
                            with rasterio.open(temp_file_path) as src:
                                arr = src.read(1).astype(np.float32)
                                nodata_val = src.nodata
                                
                                # Capture raster metadata profile from the first successful crop
                                if raster_profile is None:
                                    raster_profile = src.profile.copy()
                                
                                # Convert nodata markers to NaNs so they don't skew the math average
                                if nodata_val is not None:
                                    arr[arr == nodata_val] = np.nan
                                
                                valid_arrays.append(arr)
                                
                        except Exception as e:
                            print(f"Skipping component {filename} due to an error: {e}")
                    
                    # 3. Compute pixel-by-pixel average across available rasters
                    if valid_arrays:
                        print(f"Generating: {othr_filename} (Averaging {len(valid_arrays)} crops)")
                        
                        # Calculate mean safely ignoring NaN flags
                        with warnings.catch_warnings():
                            warnings.simplefilter("ignore", category=RuntimeWarning)
                            avg_array = np.nanmean(valid_arrays, axis=0)
                        
                        # Re-apply original nodata value back into the background pixels
                        if raster_profile and raster_profile.get('nodata') is not None:
                            final_nodata = raster_profile['nodata']
                            avg_array[np.isnan(avg_array)] = final_nodata
                        else:
                            final_nodata = -9999.0
                            avg_array[np.isnan(avg_array)] = final_nodata
                            raster_profile['nodata'] = final_nodata
                        
                        # Ensure profile settings match output format
                        raster_profile.update(dtype=rasterio.float32)
                        
                        # 4. Save the final averaged OTHR map to your main folder
                        with rasterio.open(othr_file_path, 'w', **raster_profile) as dst:
                            dst.write(avg_array.astype(raster_profile['dtype']), 1)
                    else:
                        print(f"Skipping {othr_filename}: No source crop files could be reached.")

    print("Other crop Generation Complete ---")

# Run the new OTHR averaging process
process_other_crops(GAEZ_folder)

# Part 3: Generating Land Cells

## 3.1. Generating Georeferenced Point Grid from Shapefile
Considering the resolution of GAEZ raster files, it is recommended to set spacing to 0.09 decimal degrees resulting in a detailed land and water analysis.

In [ ]:
#create a GeoDataFrame from the attributes and geometry of the shapefile
shapefile = gpd.read_file(in_path + "\\" + shp_nm)
shapefile = shapefile.to_crs(crs_WGS84)

In [ ]:
#Creating point grid
spacing = 0.09
xmin, ymin, xmax, ymax = shapefile.total_bounds

xcoords = [i for i in np.arange(xmin, xmax, spacing)]
ycoords = [i for i in np.arange(ymin, ymax, spacing)]

pointcoords = np.array(np.meshgrid(xcoords, ycoords)).T.reshape(-1, 2) 
points = gpd.points_from_xy(x=pointcoords[:,0], y=pointcoords[:,1])
grid = gpd.GeoSeries(points, crs=shapefile.crs)
grid.name = 'geometry'


#only points inside administrative boundary:
gridinside = gpd.sjoin(gpd.GeoDataFrame(grid), shapefile[['geometry']], how="inner")

#Plot georeferenced point grid
fig, ax = plt.subplots(figsize=(20, 20))
shapefile.plot(ax=ax, alpha=0.7, color="pink", edgecolor='red', linewidth=3)
grid.plot(ax=ax, markersize=30, color="blue")
gridinside.plot(ax=ax, markersize=15, color="yellow")
file_path = os.path.join(out_path, "{}_PointGrid.png".format(country_name))
plt.savefig(file_path)

## 3.2. Converting Points to Polygons

A regular grid point is created across the entire area of interest in the previous step. Georeferenced points have unique latitude and longitude. In this step,  square buffer-based polygons are created around each point. This allows further flexibility in the extraction of raster values using stats. The buffered polygon shall split "equally" the area between neighbor points; therefore, the buffer used shall be the half of the distance between two neighbor points. This, in turn depends on the location of the AoI on earth and the projection system used. 

### 3.2.1. Spatial Join
Assigning the same administrative region as defined in the GeoDataFrame to the 'cluster' column.

In [ ]:
# Calculate the centroids 
clustered_gdf = gridinside
clustered_gdf = clustered_gdf.to_crs(crs_WGS84)

In [ ]:
# Rename the columns to cluster
clustered_gdf.rename(columns={'index_right': 'cluster'}, inplace=True)

# Convert cluster column to string
clustered_gdf.cluster = clustered_gdf.cluster.astype(str).replace('0', 'NaN')

In [ ]:
# Reset the index of the left dataframe
clustered_gdf = clustered_gdf.reset_index(drop=True)
admin_name = "NAME_" + str(admin_level)

if admin_level == 0:
    # Perform the spatial join
    clustered_gdf = gpd.sjoin(clustered_gdf, shapefile[["geometry", "GID_0"]], predicate='within').drop(['cluster'], axis=1)
    
    # Rename the 'GID_0' column to 'cluster'
    clustered_gdf.rename(columns={'GID_0': 'cluster'}, inplace=True)
else:
    # Perform the spatial join
    clustered_gdf = gpd.sjoin(clustered_gdf, shapefile[["geometry", admin_name]], predicate='within').drop(['cluster'], axis=1)
    
    # Rename the 'NAME_1' column to 'cluster'
    clustered_gdf.rename(columns={admin_name: 'cluster'}, inplace=True)

# Print the first 5 rows of the joined GeoDataFrame
clustered_gdf.head(3)


In [ ]:
# create a new column based on first 3 letters of the 'cluster' column
clustered_gdf['new_cluster'] = clustered_gdf['cluster'].apply(lambda x:  x[:3]).str.upper()
clustered_gdf = clustered_gdf.rename(columns={'cluster': 'old_cluster'})
clustered_gdf = clustered_gdf.rename(columns={'new_cluster': 'cluster'})
clustered_gdf = clustered_gdf.drop(columns=['old_cluster'])
clustered_gdf.head(3)


### 3.2.2. Region Aggregation


In [ ]:
#Aggregating subnational regions based on user-defined aggregation list. The aggregated land cells of regions are represented by the Grouped Region Cluster (GRC).
if aggregate == True:
    clustered_gdf['cluster'] = clustered_gdf['cluster'].apply(lambda x: 'GRC' if x in aggregate_region else x)
    
clustered_gdf.head(3)

### 3.2.3. Generating Polygons
Creating Polygons From Georeferenced Clustered Grid Points

In [ ]:
#Buffer value used should be half the distance between two adjacent points, which in turn is dependent on the location of the Area of Interest (AoI) on Earth and the projection system being used.
buffer_value = 0.045

In [ ]:
#cap_style refers to the type of geometry generated; 3=square (see shapely documectation for more info -- https://shapely.readthedocs.io/en/stable/manual.html)

clustered_gdf['geometry'] = clustered_gdf.apply(lambda x:
                                                          x.geometry.buffer(buffer_value, cap_style=3), axis=1)

clustered_gdf.head(3)


**Note!** Several features are not classified into a cluster. While points away of the administrative borders will be cut out of the analysis, some points right next to the outer administrative borders might create inconsistency when calculating area. In the following section we are dealing with this problem.

## 3.3. Total Area Re-Estimation & Calibration

This step estimates and calibrates the area (in square km) based on the area provided by the admin layer used in the analysis (e.g. clipping). 

### 3.3.1. Area Calibration

In [ ]:
#Read admin layer as GeoDtaFrame
admin = gpd.read_file(in_path + "\\" + admin0_nm)

#Project to proper crs
admin = admin.to_crs(crs_proj)

In [ ]:
final_clustered_GAEZ_gdf = clustered_gdf
final_clustered_GAEZ_gdf.head(3)

In [ ]:
# Project datasets to proper crs
final_clustered_GAEZ_gdf_prj = final_clustered_GAEZ_gdf.to_crs(crs_proj)

In [ ]:
#add a column for area calculation
final_clustered_GAEZ_gdf_prj["sqkm"] = final_clustered_GAEZ_gdf_prj['geometry'].area/10**6

In [ ]:
def get_multiplier(estimated, official):
    if official == estimated:
        return 1
    try:
        return  official / estimated
    except ZeroDivisionError:
        return 0

In [ ]:
estimated_area = final_clustered_GAEZ_gdf_prj.sqkm.sum()
official_area = admin.geometry.area.sum()/10**6

# Estimate column multipler
multiplier = get_multiplier(estimated_area, official_area)

In [ ]:
final_clustered_GAEZ_gdf_prj.sqkm = final_clustered_GAEZ_gdf_prj.sqkm * multiplier

In [ ]:
print ("Our modelling exercise yields a total area of {0:.1f} sqkm for the country".format(estimated_area))
print ("The admin layer indicates {0:.1f} sqkm".format(official_area))
print ("After calibration the total area is set at {0:.1f} sqkm".format(final_clustered_GAEZ_gdf_prj.sqkm.sum()))

### 3.3.2. Final Check

In [ ]:
#Revert to original crs
final_clustered_GAEZ_gdf = final_clustered_GAEZ_gdf_prj.to_crs(crs_WGS84)

In [ ]:
#Final check
final_clustered_GAEZ_gdf.head(3)

### 3.3.3. Export as GeoPackage

In [ ]:
final_clustered_GAEZ_gdf.to_file(os.path.join(out_path,"{c}.gpkg".format(c=output_nm)),driver="GPKG")
print ("Part 3 complete!")


# Part 4: Geospatial Attributes Extraction to land cells

The functions employed in the fourth part extract values from TIFF-formatted GAEZ raster files, and assign them as attributes to the land cells based on their spatial locations


## 4.1. Clipping GAEZ Raster Files
The administrative boundary of the study area is used to clip the GAEZ raster files with global coverage, which leads to a reduction in the computational processing time

In [ ]:

admin = admin.to_crs(crs_WGS84)
#precipitation and land cover. 
for i in os.listdir(in_path_raster):
    with rasterio.open(os.path.join(in_path_raster, i)) as src:
        # Get the admin's CRS
        admin_crs = admin.crs
        
        # Get the geometry of the admin
        admin_geom = admin.geometry.values[0]
        
        # Crop the raster based on the admin's geometry
        out_image, out_transform = mask(src, [admin_geom], crop=True)
        
        # Update the metadata of the cropped raster
        out_meta = src.meta.copy()
        out_meta.update({
            "driver": "GTiff",
            "height": out_image.shape[1],
            "width": out_image.shape[2],
            "transform": out_transform,
            "crs": admin_crs
        })
        
        # Write the cropped raster to the output directory
        out_path_tif_crop = os.path.join(out_path_raster, i)
        with rasterio.open(out_path_tif_crop, "w", **out_meta) as dest:
            dest.write(out_image)

#GAEZ data

for i in os.listdir(GAEZ_folder):
    with rasterio.open(os.path.join(GAEZ_folder, i)) as src:
        # Get the admin's CRS
        admin_crs = admin.crs
        
        # Get the geometry of the admin
        admin_geom = admin.geometry.values[0]
        
        # Crop the raster based on the admin's geometry
        out_image, out_transform = mask(src, [admin_geom], crop=True)
        
        # Update the metadata of the cropped raster
        out_meta = src.meta.copy()
        out_meta.update({
            "driver": "GTiff",
            "height": out_image.shape[1],
            "width": out_image.shape[2],
            "transform": out_transform,
            "crs": admin_crs
        })
        
        # Write the cropped raster to the output directory
        out_path_tif_crop = os.path.join(out_path_raster, i)
        with rasterio.open(out_path_tif_crop, "w", **out_meta) as dest:
            dest.write(out_image)



## 4.2. Defining Functions

In [ ]:
# Processing Continuous/Numerical Rasters Directly on GeoDataFrame
def processing_raster_con(path, raster_filename, prefix, method, gdf):
    full_path = os.path.join(path, raster_filename)
    
    with rasterio.open(full_path) as src:
        # 1. Read array and metadata
        raster_array = src.read(1)
        affine = src.transform
        nodata_val = src.nodata

        # 2. Extract statistics 
        stats = zonal_stats(
            gdf,
            raster_array,
            affine=affine,
            nodata=nodata_val,
            stats=[method],
            all_touched=True,
            geojson_out=False
        )
    
    # 3. map results to a new column
    column_name = f"{prefix}{method}"
    gdf[column_name] = [stat.get(method) if isinstance(stat, dict) else None for stat in stats]
    
    print(f"{prefix} processing completed at {datetime.datetime.now()}")
    return gdf


# Processing Categorical/Discrete Rasters Directly on GeoDataFrame
def processing_raster_cat(path, raster_filename, prefix, gdf):
    full_path = os.path.join(path, raster_filename)
    
    with rasterio.open(full_path) as src:
        raster_array = src.read(1)
        affine = src.transform
        nodata_val = src.nodata

        stats = zonal_stats(
            gdf,
            raster_array,
            affine=affine,
            nodata=nodata_val,
            categorical=True,
            all_touched=True,
            geojson_out=False  # Massive speed boost
        )
    
    # 3. Convert list of category count dictionaries to a DataFrame and join it
    stats_df = pd.DataFrame(stats).add_prefix(prefix)
    
    # Reset index to ensure a seamless alignment match
    gdf = gdf.reset_index(drop=True)
    gdf = pd.concat([gdf, stats_df], axis=1)
    
    print(f"{prefix} processing completed at {datetime.datetime.now()}")
    return gdf

## 4.3. Collecting Raster Files

In [ ]:
# Read files with tif extension and assign their name into two list for discrete and continuous datasets
raster_files_dis = []
raster_files_con =[]

for i in os.listdir(out_path_raster):
    if ("ncb" in i) and i.endswith('.tif'):
        with rasterio.open(out_path_raster + '\\' + i) as src:
            raster_files_dis.append(i)            
    else:
        with rasterio.open(out_path_raster  + '\\' + i) as src:
            data = src.read()
            raster_files_con.append(i)

                
# keep only unique values -- Not needed but just in case there are dublicates
raster_files_con = list(set(raster_files_con))
raster_files_dis = list(set(raster_files_dis))
                
print ("We have identified {} continuous raster(s):".format(len(raster_files_con)),"\n",)
for raster in raster_files_con:
    print ( "*", raster)
    
print ("\n", "We have identified {} discrete raster(s):".format(len(raster_files_dis)),"\n",)
for raster in raster_files_dis:
    print ( "*", raster)

## 4.4. Extracting Raster Values 

In [ ]:
land_cells = final_clustered_GAEZ_gdf.copy()

### 4.4.1. Continuous Datasets (e.g. precipitation, crop evapotranspiration etc.)

In [ ]:
for raster in raster_files_con:
    prefix = raster.replace(".tif", "") + "_"
    land_cells = processing_raster_con(out_path_raster, raster, prefix, "mean", land_cells)


### 4.4.2. Categorical Datasets (e.g. Land cover type)

In [ ]:

for raster in raster_files_dis:
    # Safe replacement string logic
    prefix = raster.replace(".tif", "").replace("_ncb", "") + "_"
    land_cells = processing_raster_cat(out_path_raster, raster, prefix, land_cells)

## 4.5. Exporting the GeoDataFrame as Vector Layer

In [ ]:
# Save files 
csv_out = os.path.join(out_path, f"{result_nm}.csv")
gpkg_out = os.path.join(out_path, f"{result_nm}.gpkg")

land_cells.to_csv(csv_out, index=False)
land_cells.to_file(gpkg_out, driver="GPKG")

print(f"Part 4 complete! Files saved to: \n - {csv_out}\n - {gpkg_out}")

# Part 5: Spatial Clustering

The study area is spatially clustered based on similarities in agro-ecological potential yield using the agglomerative hierarchical clustering method 

In [ ]:

from shapely import wkt
csv_path = os.path.join(out_path, f"{result_nm}.csv")
df = pd.read_csv(csv_path)
df['geometry'] = df['geometry'].apply(wkt.loads)
land_cells = gpd.GeoDataFrame(df, geometry='geometry', crs=crs_proj)


In [ ]:
regions = list(land_cells['cluster'].unique())
display ( Markdown(' List of regions:{}'.format (regions)))

In [ ]:
#For every regional cluster, create an individual gdf according to the administrative border level.

regions_gdf = {}
regions_list = []

for i in regions:
    region_name = 'region_'+ i
    regions_list.append(region_name)
    regions_gdf[region_name] = land_cells.loc[land_cells['cluster'] == i]


In [ ]:
#Normalizing GeoDataFrame
clusters = land_cells
columns_yld = [col for col in clusters.columns if "GAEZ-V5.RES02-YLD" in col]
regions_normalized_list = []
regions_normalized_gdf = {}

for i in regions_list:
    region_normalized_name = 'normalized_'+ i
    regions_normalized_list.append(region_normalized_name)
    regions_normalized_gdf[region_normalized_name] = regions_gdf[i][columns_yld]

for i in regions_normalized_list:
    for col in regions_normalized_gdf[i]. columns:
        min_value = min(regions_normalized_gdf[i].loc[:, col])
        max_value = max(regions_normalized_gdf[i].loc[:, col])
        regions_normalized_gdf[i].loc[:, col] = (regions_normalized_gdf[i].loc[:, col] - min_value) / (max_value - min_value)

## 5.1. Defining Functions

In [ ]:
# Generating Dendrogram
def generate_dendrogram (df, name):
    
    linkage_matrix_normalized = linkage(df.values, method='ward', metric='euclidean')
    dendrogram(linkage_matrix_normalized)
    plt.title('Dendrogram '+ name)
    
    file_path=os.path.join(out_path_dendrogram, name +"_" + "{}_Dendrogram_Yield".format(country_name))
    plt.savefig(file_path)
    plt.close()
    #plt.show()

In [ ]:
# Elbow Method

os.environ["OMP_NUM_THREADS"] = "1"

def generate_elbow_graph (df, name):
    
    wcss = [] # Initialize a list to store the within-cluster sum of squares
    
    for k in range(1, 10): # Initial numbers of clusters to calculate WCSS
        kmeans = KMeans(n_clusters=k, random_state=0, n_init='auto')
        kmeans.fit(df)
        wcss.append(kmeans.inertia_)  # Calculate WCSS for each k
    
    
    # Plot the elbow graph values to identify the elbow point (optimum number of clusters)
    plt.figure(figsize=(8, 10))
    plt.plot(range(1, 10), wcss, marker='o', linestyle='-', color='b')
    plt.title('Elbow Graph for Optimal Number of Clusters ' + name)
    plt.xlabel('Number of Clusters (k)')
    plt.ylabel('WCSS')
    plt.grid()
    
    file_path=os.path.join(out_path_elbow, name +"_"+ "{}_elbow_Yield".format(country_name))
    plt.savefig(file_path)
    plt.close()
    #plt.show()



In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
# Define agglomerative clustering
def agglomerative_clustering(gdf, df, n_clusters, name):
    if len(df) < n_clusters:
        n_clusters = 1

    agglo_clustering = AgglomerativeClustering(n_clusters=n_clusters, linkage='ward')
    gdf['clusters_yield'] = agglo_clustering.fit_predict(df) + 1

    labels = sorted(gdf['clusters_yield'].unique())
    
    # Note: Modern matplotlib prefers plt.get_cmap over importing get_cmap directly
    cmap = plt.get_cmap('viridis', len(labels))

    ax = gdf.plot(
        column='clusters_yield',
        categorical=True,
        cmap=cmap,
        legend=False,
        figsize=(10, 8),
        linewidth=0.1,
        edgecolor='black'
    )

    handles = [
        mpatches.Patch(color=cmap(i), label=f'Cluster {label}')
        for i, label in enumerate(labels)
    ]
    
    # FIX: Move the legend outside the bounding box to the right side
    ax.legend(
        handles=handles, 
        title='Cluster', 
        loc='upper left',          # Anchor the top-left corner of the legend box
        bbox_to_anchor=(1.02, 1),  # Coordinates (X, Y) relative to map. 1.02 is just past the right edge.
        borderaxespad=0
    )

    # Note: Ensure 'Full_name' is defined globally or passed into the function
    plt.title('Clustered yield ' + Full_name)
    
    file_path_new = os.path.join(out_path_clustering, f"{name}_{country_name}_Clustered_Yield.png")
    
    # FIX: bbox_inches='tight' prevents the legend from being clipped in the saved image
    plt.savefig(file_path_new, bbox_inches='tight', dpi=300)
    plt.show()

    gdf.to_file(os.path.join(out_path_clustering, f"{name}_{country_name}_clustered_data.gpkg"), driver="GPKG")
    gdf.to_csv(os.path.join(out_path_clustering, f"{name}_{country_name}_clustered_data.csv"))

In [ ]:
for i in regions_normalized_list:
    regions_normalized_gdf[i].fillna(0, inplace=True)

In [ ]:
# Generate dendrogram

for i in regions_normalized_list:
    generate_dendrogram(regions_normalized_gdf[i], i)

In [ ]:
# generate elbow graph for each region  

with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=UserWarning)
    #if n_samples < n_clusters:
        #n_clusters = 1
    for i in regions_normalized_list:
        generate_elbow_graph (regions_normalized_gdf[i],i)

In [ ]:
# # generate elbow graph for each region  
with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=UserWarning)
        
    for i in regions_normalized_list:
        n_samples = len(regions_normalized_gdf[i])
        if n_samples < 2:
            print(f"Skipping elbow graph for {i}: not enough samples ({n_samples})")
            continue
        max_k = min(10, n_samples)
        wcss = []
        for k in range(1, max_k):
            kmeans = KMeans(n_clusters=k, random_state=0, n_init='auto')
            kmeans.fit(regions_normalized_gdf[i])
            wcss.append(kmeans.inertia_)
        plt.figure(figsize=(8, 10))
        plt.plot(range(1, max_k), wcss, marker='o', linestyle='-', color='b')
        plt.title('Elbow Graph for Optimal Number of Clusters ' + i)
        plt.xlabel('Number of Clusters (k)')
        plt.ylabel('WCSS')
        plt.grid()
        file_path=os.path.join(out_path_elbow, i +"_"+ "{}_elbow_Yield".format(country_name))
        plt.savefig(file_path)
        plt.close()

## 5.2. User Input for Spatial Clustering 
Determine the optimal number of clusters using the generated dendrogram and elbow graphs that have been generated and stored within the 'Data\output' directory.

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=pd.errors.SettingWithCopyWarning)
    
    # Choose optimum number of clusters
    default_cluster = 1  # Default number of clusters
    
    for i in regions_normalized_list:
        user_input = input("Enter the optimum number of clusters in {} (or press Enter for default): ".format(i))

        # Use the user input or the default value
        number_cluster = int(user_input.strip()) if user_input else default_cluster

        for j in regions_list:
            if j in i:
                # Perform agglomerative hierarchical clustering
                agglomerative_clustering (regions_gdf[j], regions_normalized_gdf[i] , number_cluster, j)


# Part 6: Statistics Calculation

This part calculates summary statistics for the generated clusters. There outputs include:

* Tabular summaries (.csv format) grouped by cluster
* CLEWs input files (.csv format) divided by farming system 



## 6.1. Calculating Cluster Summaries

In [ ]:
# Collect names of attributes assigned to the clusters.
origin_list_of_cols = list(final_clustered_GAEZ_gdf.columns)
final_list_of_cols = list(land_cells.columns)

In [ ]:
# Land cover area estimator
def calc_LC_sqkm(df, col_list):
    """ 
    This function takes the df where the LC type for different classes is provided per location (row).
    It adds all pixels per location; then is calculates the ratio of LC class in each location (% of total).
    Finally is estimates the area per LC type in each location by multiplying with the total area each row represents.
    
    INPUT: 
    df -> Pandas dataframe with LC type classification 
    col_list -> list of columns to include in the summary (e.g. LC1-LC11)
    
    OUTPUT: Updated dataframe with estimated area (sqkm) of LC types per row
    """
    df["LC_sum"] = df[col_list].sum(axis=1)
    for col in col_list:
        df[col] = df[col]/df["LC_sum"]*df["sqkm"]
    
    return df


In [ ]:
# Identify land cover related columns
landCover_cols = []
for col in final_list_of_cols:
    if "LCType" in col:
        landCover_cols.append(col)
if not landCover_cols:
    print ("There is not any Land Cover associated column in the dataframe; please revise")
else:
    pass

In [ ]:
data_gdf_LCsqkm_list = []
data_gdf_LCsqkm = {}

for i in regions_list:
    data_gdf_LCsqkm_name = 'data_gdf_LCsqkm_'+ i
    data_gdf_LCsqkm_list.append(data_gdf_LCsqkm_name)
    data_gdf_LCsqkm [data_gdf_LCsqkm_name]= calc_LC_sqkm (regions_gdf[i], landCover_cols)

In [ ]:
print(regions_list)

## 6.2 Unit Conversion  

In [ ]:
#new
# Calculate summary statistics for other than land cover attribute columns
data_gdf_stat = data_gdf_LCsqkm

# Define the conversion factor for CLEWs modelling
#Potential yield unit conversion from kg DW/ha to million tonnes per 1000 sqkm
factor1 = 0.0001 

#Other parameter unit conversion from millimeter to BCM per 1000 sqkm
factor2 = 0.001

for i in data_gdf_LCsqkm_list:
    for col in data_gdf_stat[i].columns:
        if col.startswith("GAEZ-V5.RES02-YLD"):
            factor = factor1
        elif col.startswith("GAEZ-V5.RES02-ETA"):
            factor = factor2
        elif col.startswith("GAEZ-V5.RES02-WDE"):
            factor = factor2
        else:
            continue

        data_gdf_stat[i].loc[:, col] *= factor


In [ ]:
print(data_gdf_LCsqkm[i].columns)

In [ ]:
sum_cols = [x for x in final_list_of_cols if x not in origin_list_of_cols]
sum_cols = [x for x in sum_cols if x not in landCover_cols]
for col_to_remove in ["id", "LC_sum"]:
    if col_to_remove in sum_cols:
        sum_cols.remove(col_to_remove)


## 6.3. Land Cover and Area Statistics

In [ ]:
group_lc = {}

for key, gdf in data_gdf_stat.items():
    # Group by 'clusters_yield'
    clusters = gdf.groupby(['clusters_yield'])
    
    
    clusters_lc = clusters[landCover_cols].sum().merge(clusters["sqkm"].sum().reset_index(name="sqkm"), on="clusters_yield").round(decimals=1)
    clusters_lc = clusters_lc.sort_values(ascending=True, by='clusters_yield').reset_index(drop=True)
    name = key[-3:]
    
    #Export land cover stats to csv
    clusters_lc.to_csv(os.path.join(summary_stats_path,"{}_LandCover_byCluster_summary.csv".format(name)),index_label='cluster')

    # Display summary statistics
    display(Markdown('#### Cluster summary statistics for area and land cover in {}'.format(name)))
    display(Markdown(' **Total area:** {:0.1f} sq.km'.format(clusters_lc.sqkm.sum())))
    display(clusters_lc)

    # Store the result in the group_dic dictionary
    group_lc[key] = clusters_lc



### 6.4. Calculating the Average of Additional Crops

In [ ]:
#Add new column contain average value of five to ten in the top 10 crops 
def additional_stat_group(parameter):
    selected_group = [col for col in additional_stat_table_group.columns if parameter in col]

    selected_group = additional_stat_table_group.loc[:, selected_group]
    
    Irrigation_Low_group = selected_group.loc[:, [a for a in selected_group.columns if 'Irrigation Low' in a]]
    Low_group = round(Irrigation_Low_group.mean(axis=1),4)
    clusters_stat['OTH'+' '+ parameter +' '+'Irrigation Low_mean'] = Low_group
    
    Irrigation_High_group = selected_group.loc[:, [a for a in selected_group.columns if 'Irrigation High' in a]]
    High_group = round(Irrigation_High_group.mean(axis=1),4)            
    clusters_stat['OTH'+' '+ parameter +' '+ 'Irrigation High_mean'] = High_group    


    Rain_fed_Low_group = selected_group.loc[:, [a for a in selected_group.columns if 'Rain-fed Low' in a]]   
    Rain_Low = round(Rain_fed_Low_group.mean(axis=1),4)            
    clusters_stat['OTH'+' '+ parameter +' '+ 'Rain-fed Low_mean'] = Rain_Low 
                
    Rain_fed_High_group = selected_group.loc[:, [a for a in selected_group.columns if 'Rain-fed High' in a]]         
    Rain_High = round(Rain_fed_High_group.mean(axis=1),4)               
    clusters_stat['OTH'+' '+ parameter +' '+ 'Rain-fed High_mean'] =Rain_High  


### 6.5. Other Variable Statistics

In [ ]:
group_stat = {}
for key, gdf in data_gdf_stat.items():
    # Group by 'clusters_yield'
    clusters = gdf.groupby(['clusters_yield'])
    
    clusters_stat = clusters[sum_cols].mean().round(decimals = 4)

    
    additional_crop_stat_group = [col for col in clusters_stat.columns if any(a in col for a in other_crop_name)]
    additional_stat_table_group = clusters_stat.loc[:, additional_crop_stat_group].copy()
    clusters_stat = clusters_stat.drop(additional_stat_table_group, axis=1)
    
    additional_stat_group('yld')
    additional_stat_group('cwd')
    additional_stat_group('evt')
    
    name = key[-3:]
    
    display(Markdown('#### Cluster summary statistics for other variables in {}'.format(name)))
    display(clusters_stat)

    
    group_stat[key] = clusters_stat
    

## 6.6. Generate clewsy-compatible Statistics

In [ ]:
for key, gdf in group_stat.items():    
    clusters_other = gdf
    
    name = key[-3:]

    # generating the crop potential yeild csv files
    yld_columns = [col for col in clusters_other.columns if 'GAEZ-V5.RES02-YLD' in col]
    yld_df = clusters_other[yld_columns]

    # Name correction according clewsy format
    yld_rename = {col: col.replace(' GAEZ-V5.RES02-YLD', '').replace('_mean', '') for col in yld_df.columns}
    yld_df = yld_df.rename(columns=yld_rename)
    empty_yld = pd.DataFrame(columns=['', '', '', '','', '', '', '', ''],index= yld_df.index)
    combined_yld = pd.concat([empty_yld, yld_df], axis=1)

    #Store as CSV file
    combined_yld.to_csv(os.path.join(summary_stats_path,"clustering_results_{}.csv".format(name)), index= True, index_label='cluster')


    # generating crop evapotranspiration csv files
    evt_columns = [col for col in clusters_other.columns if 'GAEZ-V5.RES02-ETA' in col]
    evt_df = clusters_other[evt_columns]

    # Name correction
    evt_rename = {col: col.replace(' GAEZ-V5.RES02-ETA', '').replace('_mean', '') for col in evt_df.columns}
    evt_df = evt_df.rename(columns=evt_rename)
    empty_evt = pd.DataFrame(columns=['', '', '', '','', '', '', '', ''],index= evt_df.index)
    combined_evt = pd.concat([empty_evt, evt_df], axis=1)

    combined_evt.to_csv(os.path.join(summary_stats_path,"clustering_results_evt_{}.csv".format(name)), index= True, index_label='cluster')



    #generating crop water deficit csv files 
    cwd_columns = [col for col in clusters_other.columns if 'GAEZ-V5.RES02-WDE' in col]
    cwd_df = clusters_other[cwd_columns]

    # Name correction
    cwd_rename = {col: col.replace(' GAEZ-V5.RES02-WDE', '').replace('_mean', '') for col in cwd_df.columns}
    cwd_df = cwd_df.rename(columns=cwd_rename)
    empty_cwd = pd.DataFrame(columns=['', '', '', '','', '', '', '', ''],index= cwd_df .index)
    combined_cwd = pd.concat([empty_cwd, cwd_df], axis=1)

    combined_cwd.to_csv(os.path.join(summary_stats_path,"clustering_results_cwd_{}.csv".format(name)), index= True, index_label='cluster')


    # generating precipitation csv files
    prc_columns = [col for col in clusters_other.columns if 'prc' in col]
    prc_df = round(clusters_other[prc_columns]*0.001, 3) #convert to BCM per 1000 sqkm

    prc_rename = {col: col.replace(' prc', '').replace('_mean', '') for col in prc_df.columns}
    prc_df = prc_df.rename(columns=prc_rename)

    prc_df.to_csv(os.path.join(summary_stats_path,"clustering_results_prc_{}.csv".format(name)), index= True, index_label='cluster')


    #Export summary stats to csv
    clusters_other.to_csv(os.path.join(summary_stats_path,"{}_Parameter_byCluster_summary.csv".format(name)))


## 6.7. Create CLEWs datasets

In [ ]:
#Irrigation High CLEWs input generation

# File paths
clustering_results_path = os.path.join(summary_stats_path, "clustering_results_GRC.csv")
cwd_path = os.path.join(summary_stats_path, "clustering_results_cwd_GRC.csv")
evt_path = os.path.join(summary_stats_path, "clustering_results_evt_GRC.csv")
prc_path = os.path.join(summary_stats_path, "clustering_results_prc_GRC.csv")

output_folder = os.path.join(summary_stats_path, "CLEWs inputs")
os.makedirs(output_folder, exist_ok=True)

def extract_data(df, value_name):
    '''
    This function extracts crop codes and filters the desired input.
    '''
    df = df.copy()
    id_vars = ["cluster"]
    value_vars = [c for c in df.columns if c not in id_vars]

    melted = df.melt(id_vars=id_vars, value_vars=value_vars, var_name="source", value_name=value_name)
    pattern = re.compile(r"\.([A-Z]{4})\.([A-Z]{4})$")

    def parse_source(src):
        match = pattern.search(src)
        if match:
            crop_code = match.group(1)
            input_code = match.group(2)
            return crop_code, input_code
        return None, None

    parsed = melted["source"].apply(parse_source)
    melted[["CROP", "INPUT"]] = pd.DataFrame(parsed.tolist(), index=melted.index)

    # Keep only one input type
    melted = melted[melted["INPUT"].str.startswith("HI", na=False)]

    return melted[["cluster", "CROP", value_name]]

# Load data
prc_df = pd.read_csv(prc_path)
cwd_df = pd.read_csv(cwd_path)
evt_df = pd.read_csv(evt_path)
yld_df = pd.read_csv(clustering_results_path)

# Extract only the high-irrigated columns
cwd_long = extract_data(cwd_df, "CWD")
evt_long = extract_data(evt_df, "EVP")
yld_long = extract_data(yld_df, "YIELD")

# Merge the crop-specific variables
result = yld_long.merge(cwd_long, on=["cluster", "CROP"], how="outer")
result = result.merge(evt_long, on=["cluster", "CROP"], how="outer")

# Add precipitation for the cluster
result = result.merge(prc_df, on="cluster", how="left")

# Keep / rename columns
result = result[["cluster", "CROP", "precipitation", "CWD", "EVP", "YIELD"]]
result.columns = ["CLUSTER", "CROP", "PRC", "CWD", "EVP", "YIELD"]

output_path = os.path.join(output_folder, "Irrigation High.csv")
result.to_csv(output_path, index=False)

print("Saved:", output_path)

In [ ]:
#Irrigation Low CLEWs input generation

# File paths
clustering_results_path = os.path.join(summary_stats_path, "clustering_results_GRC.csv")
cwd_path = os.path.join(summary_stats_path, "clustering_results_cwd_GRC.csv")
evt_path = os.path.join(summary_stats_path, "clustering_results_evt_GRC.csv")
prc_path = os.path.join(summary_stats_path, "clustering_results_prc_GRC.csv")

output_folder = os.path.join(summary_stats_path, "CLEWs inputs")
os.makedirs(output_folder, exist_ok=True)

def extract_data(df, value_name):
    '''
    This function extracts crop codes and filters the desired input.
    '''
    df = df.copy()
    id_vars = ["cluster"]
    value_vars = [c for c in df.columns if c not in id_vars]

    melted = df.melt(id_vars=id_vars, value_vars=value_vars, var_name="source", value_name=value_name)
    pattern = re.compile(r"\.([A-Z]{4})\.([A-Z]{4})$")

    def parse_source(src):
        match = pattern.search(src)
        if match:
            crop_code = match.group(1)
            input_code = match.group(2)
            return crop_code, input_code
        return None, None

    parsed = melted["source"].apply(parse_source)
    melted[["CROP", "INPUT"]] = pd.DataFrame(parsed.tolist(), index=melted.index)

    # Keep only one input type
    melted = melted[melted["INPUT"].str.startswith("LI", na=False)]

    return melted[["cluster", "CROP", value_name]]

# Load data
prc_df = pd.read_csv(prc_path)
cwd_df = pd.read_csv(cwd_path)
evt_df = pd.read_csv(evt_path)
yld_df = pd.read_csv(clustering_results_path)

# Extract only the low-irrigated columns
cwd_long = extract_data(cwd_df, "CWD")
evt_long = extract_data(evt_df, "EVP")
yld_long = extract_data(yld_df, "YIELD")

# Merge the crop-specific variables
result = yld_long.merge(cwd_long, on=["cluster", "CROP"], how="outer")
result = result.merge(evt_long, on=["cluster", "CROP"], how="outer")

# Add precipitation for the cluster
result = result.merge(prc_df, on="cluster", how="left")

# Keep / rename columns
result = result[["cluster", "CROP", "precipitation", "CWD", "EVP", "YIELD"]]
result.columns = ["CLUSTER", "CROP", "PRC", "CWD", "EVP", "YIELD"]

output_path = os.path.join(output_folder, "Irrigation Low.csv")
result.to_csv(output_path, index=False)

print("Saved:", output_path)

In [ ]:
#Rain-fed High CLEWs input generation

# File paths
clustering_results_path = os.path.join(summary_stats_path, "clustering_results_GRC.csv")
cwd_path = os.path.join(summary_stats_path, "clustering_results_cwd_GRC.csv")
evt_path = os.path.join(summary_stats_path, "clustering_results_evt_GRC.csv")
prc_path = os.path.join(summary_stats_path, "clustering_results_prc_GRC.csv")

output_folder = os.path.join(summary_stats_path, "CLEWs inputs")
os.makedirs(output_folder, exist_ok=True)

def extract_data(df, value_name):
    '''
    This function extracts crop codes and filters the desired input.
    '''
    df = df.copy()
    id_vars = ["cluster"]
    value_vars = [c for c in df.columns if c not in id_vars]

    melted = df.melt(id_vars=id_vars, value_vars=value_vars, var_name="source", value_name=value_name)
    pattern = re.compile(r"\.([A-Z]{4})\.([A-Z]{4})$")

    def parse_source(src):
        match = pattern.search(src)
        if match:
            crop_code = match.group(1)
            input_code = match.group(2)
            return crop_code, input_code
        return None, None

    parsed = melted["source"].apply(parse_source)
    melted[["CROP", "INPUT"]] = pd.DataFrame(parsed.tolist(), index=melted.index)

    # Keep only one input type
    melted = melted[melted["INPUT"].str.startswith("HR", na=False)]

    return melted[["cluster", "CROP", value_name]]

# Load data
prc_df = pd.read_csv(prc_path)
cwd_df = pd.read_csv(cwd_path)
evt_df = pd.read_csv(evt_path)
yld_df = pd.read_csv(clustering_results_path)

# Extract only the rain-fed columns
cwd_long = extract_data(cwd_df, "CWD")
evt_long = extract_data(evt_df, "EVP")
yld_long = extract_data(yld_df, "YIELD")

# Merge the crop-specific variables
result = yld_long.merge(cwd_long, on=["cluster", "CROP"], how="outer")
result = result.merge(evt_long, on=["cluster", "CROP"], how="outer")

# Add precipitation for the cluster
result = result.merge(prc_df, on="cluster", how="left")

# Keep / rename columns
result = result[["cluster", "CROP", "precipitation", "CWD", "EVP", "YIELD"]]
result.columns = ["CLUSTER", "CROP", "PRC", "CWD", "EVP", "YIELD"]

output_path = os.path.join(output_folder, "Rain-fed High.csv")
result.to_csv(output_path, index=False)

print("Saved:", output_path)

In [ ]:
#Low Rain-fed CLEWs input generation

# File paths
clustering_results_path = os.path.join(summary_stats_path, "clustering_results_GRC.csv")
cwd_path = os.path.join(summary_stats_path, "clustering_results_cwd_GRC.csv")
evt_path = os.path.join(summary_stats_path, "clustering_results_evt_GRC.csv")
prc_path = os.path.join(summary_stats_path, "clustering_results_prc_GRC.csv")

output_folder = os.path.join(summary_stats_path, "CLEWs inputs")
os.makedirs(output_folder, exist_ok=True)

def extract_data(df, value_name):
    '''
    This function extracts crop codes and filters the desired input.
    '''
    df = df.copy()
    id_vars = ["cluster"]
    value_vars = [c for c in df.columns if c not in id_vars]

    melted = df.melt(id_vars=id_vars, value_vars=value_vars, var_name="source", value_name=value_name)
    pattern = re.compile(r"\.([A-Z]{4})\.([A-Z]{4})$")

    def parse_source(src):
        match = pattern.search(src)
        if match:
            crop_code = match.group(1)
            input_code = match.group(2)
            return crop_code, input_code
        return None, None

    parsed = melted["source"].apply(parse_source)
    melted[["CROP", "INPUT"]] = pd.DataFrame(parsed.tolist(), index=melted.index)

    # Keep only one input type
    melted = melted[melted["INPUT"].str.startswith("LR", na=False)]

    return melted[["cluster", "CROP", value_name]]

# Load data
prc_df = pd.read_csv(prc_path)
cwd_df = pd.read_csv(cwd_path)
evt_df = pd.read_csv(evt_path)
yld_df = pd.read_csv(clustering_results_path)

# Extract only the low rain-fed columns
cwd_long = extract_data(cwd_df, "CWD")
evt_long = extract_data(evt_df, "EVP")
yld_long = extract_data(yld_df, "YIELD")

# Merge the crop-specific variables
result = yld_long.merge(cwd_long, on=["cluster", "CROP"], how="outer")
result = result.merge(evt_long, on=["cluster", "CROP"], how="outer")

# Add precipitation for the cluster
result = result.merge(prc_df, on="cluster", how="left")

# Keep / rename columns
result = result[["cluster", "CROP", "precipitation", "CWD", "EVP", "YIELD"]]
result.columns = ["CLUSTER", "CROP", "PRC", "CWD", "EVP", "YIELD"]

output_path = os.path.join(output_folder, "Low Rain-fed.csv")
result.to_csv(output_path, index=False)

print("Saved:", output_path)

In [ ]:
print ("Part 6 - and with that the analysis - completed!")
print ("Total elapsed time: {}".format(time.strftime("%H:%M:%S", time.gmtime(time.time() - start_time))))

# Part 7: Creating shapefiles for each cluster

In [ ]:
#Plotting the clusters' boundaries

clustered_gdf_path = Path(out_path_clustering) / f"region_GRC_{country_name}_ZMB_clustered_data.gpkg"
clustered_gdf = gpd.read_file(clustered_gdf_path)

dissolved = clustered_gdf.dissolve(by='clusters_yield')

boundaries = dissolved.geometry.boundary
boundaries_gdf = gpd.GeoDataFrame(geometry=boundaries, crs=clustered_gdf.crs)

boundaries_gdf.to_file(Path(out_path_clustering) / f"region_GRC_{country_name}_ZMB_cluster_boundaries.gpkg", driver="GPKG")

boundaries_gdf.plot()

In [ ]:
#create cluster shapefile

def export_cluster_shapefiles(out_path_clustering, gpkg_name=f"region_GRC_{country_name}_ZMB_clustered_data.gpkg", target_folder="clusters", crs_target=None):
    out_path_clustering = Path(out_path_clustering)
    gpkg_path = out_path_clustering / gpkg_name
    if not gpkg_path.exists():
        raise FileNotFoundError(f"Could not find: {gpkg_path}")

    gdf = gpd.read_file(gpkg_path)

    if "clusters_yield" not in gdf.columns:
        raise KeyError("'clusters_yield' not found in the GeoDataFrame columns")

    if crs_target is not None:
        if gdf.crs is None:
            gdf = gdf.set_crs(crs_target)
        elif gdf.crs != crs_target:
            gdf = gdf.to_crs(crs_target)

    clusters_dir = out_path_clustering / target_folder
    clusters_dir.mkdir(parents=True, exist_ok=True)

    saved_files = []
    for cluster_value, cluster_gdf in gdf.groupby("clusters_yield"):
        cluster_filename = clusters_dir / f"cluster{cluster_value}.shp"
        cluster_gdf.to_file(cluster_filename, driver="ESRI Shapefile")
        saved_files.append(cluster_filename)

    return saved_files


saved = export_cluster_shapefiles(
    out_path_clustering=out_path_clustering,
    gpkg_name=f"region_GRC_{country_name}_ZMB_clustered_data.gpkg",
    target_folder="clusters",
    crs_target=crs_proj
)

print("Saved shapefiles:", saved)

In [ ]:
cluster_folder = Path(out_path_clustering) / "clusters"

for shp_path in cluster_folder.glob("*.shp"):
    gdf = gpd.read_file(shp_path)
    fig, ax = plt.subplots(figsize=(10, 10))
    gdf.plot(ax=ax, edgecolor="black")
    ax.set_title(shp_path.name)
    ax.axis("off")
    plt.show()